In [13]:
""" 0. set-up part:  import necessary libraries and set up environment """

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from collections import Counter, defaultdict
import numpy as np
import math
import copy
import itertools
import matplotlib.pyplot as plt
import matplotlib as mpl

import joblib
from joblib import Parallel, delayed
from threading import Thread

import os
import pickle
import time

import operator
from functools import reduce
import json
import cProfile

import gensim
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

import tomotopy as tp

# download nltk data once time
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('omw-1.4')
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

#  chinese character support in matplotlib
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS' 'SimHei' 'DejaVu Sans']  
plt.rcParams['axes.unicode_minus'] = False

In [14]:
""" 1.1 Data Preprocessing: load data, clean text, lemmatization, remove low-frequency words"""

# Map POS tags to WordNet format， Penn Treebank annotation: fine-grained (45 tags), WordNet annotation: coarse-grained (4 tags: a, v, n, r)
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return 'a'  # 形容词
    elif treebank_tag.startswith('V'):
        return 'v'  # 动词
    elif treebank_tag.startswith('N'):
        return 'n'  # 名词
    elif treebank_tag.startswith('R'):
        return 'r'  # 副词
    else:
        return 'n'  # 默认名词

# Text cleaning and lemmatization preprocessing function
def clean_and_lemmatize(text):
    if pd.isnull(text):
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Remove non-alphabetic characters using regex
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    pos_tags = pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(pos)) for w, pos in pos_tags]
    return lemmatized  

#-----------------Load data----------------
data = pd.read_excel('./data/raw/papers_CM.xlsx', usecols=['PaperID', 'Abstract', 'Keywords', 'Year'])

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# clean and lemmatize the abstracts
data['Lemmatized_Tokens'] = data['Abstract'].apply(clean_and_lemmatize)

# count word frequencies
all_tokens = [word for tokens in data['Lemmatized_Tokens'] for word in tokens]
word_counts = Counter(all_tokens)

# set a minimum frequency threshold for valid words
min_freq = 10
valid_words = set([word for word, freq in word_counts.items() if freq >= min_freq])

# remove rare words based on frequency threshold
def remove_rare_words(tokens):
    return [word for word in tokens if word in valid_words]

data['Filtered_Tokens'] = data['Lemmatized_Tokens'].apply(remove_rare_words)

# join tokens back into cleaned abstracts
data['Cleaned_Abstract'] = data['Filtered_Tokens'].apply(lambda x: " ".join(x))

# create a cleaned DataFrame with relevant columns
cleaned_data = data[['PaperID', 'Year', 'Cleaned_Abstract']]
cleaned_data = cleaned_data[~(cleaned_data['PaperID'] == 57188)] # this paper has no abstract
cleaned_data = cleaned_data.reset_index(drop=True) 
cleaned_data.insert(0, 'Document_ID', range(len(cleaned_data))) 
abstract_list = cleaned_data['Cleaned_Abstract'].apply(lambda x: x.split()).tolist()

corpus = {doc_id: abstract_list for doc_id, abstract_list in enumerate(abstract_list)}
# cleaned_data.to_csv('./data/processed/cleaned_data.xlsx', index=False, encoding='utf-8-sig')

In [15]:
# ===== Enhanced Coherence Calculation Function (Supports Multiple Metrics Including NPMI) =====
def calculate_multiple_coherence_metrics(mdl, corpus_docs, metrics=['c_v', 'c_npmi'], fast_mode=True, top_n=5):
    """
    Enhanced coherence calculation function - supports multiple coherence metrics including NPMI
    
    Supported coherence metrics:
    - c_v: Vector space-based coherence (default)
    - c_npmi: Normalized Pointwise Mutual Information (NPMI)
    
    Supported model types:
    - Single-layer models: LDA, CTM
    - Hierarchical models: hLDA, PAM, hPAM
    - Non-parametric models: HDP
    
    Returns: dict containing various coherence metrics
    """
    try:
        docs = list(corpus_docs)
        dictionary = Dictionary(docs)
        
        # Check if it is a hierarchical model
        model_type_str = str(type(mdl))
        is_hierarchical = hasattr(mdl, 'depth') or 'HLDA' in model_type_str or 'PAM' in model_type_str
        
        if is_hierarchical:
            # Hierarchical model: calculate coherence by level, weighted average
            metrics_results = {metric: [] for metric in metrics}
            layer_weights = []
            
            try:
                for level in range(getattr(mdl, 'depth', 3)):
                    level_topics = []
                    level_doc_count = 0
                    
                    # Iterate through all nodes of this level
                    for k in range(getattr(mdl, 'k', 100)):
                        try:
                            topic_words = mdl.get_topic_words(k, top_n=top_n)
                            if topic_words:
                                words = [word for word, prob in topic_words]
                                level_topics.append(words)
                                level_doc_count += 1
                        except:
                            continue
                    
                    if level_topics:
                        # Calculate coherence for each metric
                        for metric in metrics:
                            try:
                                cm = CoherenceModel(
                                    topics=level_topics,
                                    texts=docs,
                                    dictionary=dictionary,
                                    coherence=metric,
                                    processes=1
                                )
                                score = cm.get_coherence()
                                if score and not math.isnan(score):
                                    metrics_results[metric].append(score)
                                else:
                                    metrics_results[metric].append(0.1)
                            except Exception as e:
                                print(f"Hierarchical model {metric} calculation warning: {e}")
                                metrics_results[metric].append(0.1)
                        
                        layer_weights.append(level_doc_count)
            except:
                pass
            
            # Calculate weighted average
            final_results = {}
            for metric in metrics:
                if metrics_results[metric] and layer_weights:
                    total_weight = sum(layer_weights)
                    if total_weight > 0:
                        weighted_avg = sum(score * w for score, w in zip(metrics_results[metric], layer_weights)) / total_weight
                        final_results[metric] = weighted_avg
                    else:
                        final_results[metric] = 0.1
                else:
                    final_results[metric] = 0.1
            
            return final_results
        
        # Single-layer model: calculate coherence for all topics
        num_topics = getattr(mdl, 'k', None) or getattr(mdl, 'num_topics', None) or 100
        topics = []
        
        for k in range(num_topics):
            try:
                topic_words = mdl.get_topic_words(k, top_n=top_n)
                if topic_words:
                    words = [word for word, prob in topic_words]
                    topics.append(words)
            except:
                continue
        
        if not topics:
            return {metric: 0.1 for metric in metrics}
        
        # Calculate coherence for each metric
        final_results = {}
        for metric in metrics:
            try:
                cm = CoherenceModel(
                    topics=topics,
                    texts=docs,
                    dictionary=dictionary,
                    coherence=metric,
                    processes=1
                )
                score = cm.get_coherence()
                final_results[metric] = score if score and not math.isnan(score) else 0.1
            except Exception as e:
                print(f"Single-layer model {metric} calculation warning: {e}")
                final_results[metric] = 0.1
        
        return final_results
        
    except Exception as e:
        print(f"Coherence calculation error: {e}")
        return {metric: 0.1 for metric in metrics}

In [16]:
def calculate_renyi_entropy_unweighted(model, alpha=2):
    """
    Calculate the Renyi entropy for all topics (unweighted version, direct average)
    - model: topic model object (e.g., tomotopy LDA/CTM/PAM/hLDA)
    - alpha: order of Renyi entropy (commonly 2)
    Returns: the average of Renyi entropies for all topics
    """
    # ...existing code...
    import numpy as np
    entropies = []
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    for k in range(num_topics):
        try:
            topic_probs = np.array([prob for word, prob in model.get_topic_words(k, top_n=-1)])
            topic_probs = topic_probs / topic_probs.sum()
            if len(topic_probs) > 0:
                renyi = (1/(1-alpha)) * np.log(np.sum(topic_probs**alpha))
                entropies.append(renyi)
            else:
                entropies.append(0)
        except:
            entropies.append(0)
    if entropies:
        return float(np.mean(entropies))
    else:
        return 0.0

In [17]:
# ===== 加权Renyi熵计算函数 (Weighted Renyi Entropy) =====
def get_topic_doc_counts(model, threshold=0.01):
    """
    获取每个主题覆盖的文档数（即每个主题出现在了多少个文档中）
    """
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    topic_doc_counts = [0] * num_topics
    for doc in model.docs:
        try:
            topic_dist = doc.get_topic_dist()
        except:
            continue
        for k, prob in enumerate(topic_dist):
            if prob > threshold:
                topic_doc_counts[k] += 1
    return topic_doc_counts

def calculate_weighted_renyi_entropy(model, alpha=2):
    """
    按主题覆盖的文档数加权的Renyi熵
    """
    entropies = []
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    for k in range(num_topics):
        try:
            topic_probs = np.array([prob for word, prob in model.get_topic_words(k, top_n=-1)])
            if topic_probs.sum() > 0:
                topic_probs = topic_probs / topic_probs.sum()
                renyi = (1/(1-alpha)) * np.log(np.sum(topic_probs**alpha))
                entropies.append(renyi)
            else:
                entropies.append(0)
        except:
            entropies.append(0)
    
    # 按主题的文档覆盖数进行加权
    topic_doc_counts = get_topic_doc_counts(model)
    total_docs_covered = sum(topic_doc_counts)
    
    if total_docs_covered > 0 and len(entropies) == len(topic_doc_counts):
        return np.average(entropies, weights=topic_doc_counts)
    elif entropies:
        return np.mean(entropies) # 如果加权失败，则返回普通平均值
    else:
        return 0.0

In [18]:
from scipy.spatial.distance import jensenshannon
import numpy as np

def calculate_topic_diversity_jsd(model, top_n=25):
    """
    计算所有主题对之间的平均JSD（Jensen-Shannon Divergence），以衡量主题多样性。
    JSD值域为[0, 1]，值越高代表主题间差异越大，多样性越好。

    - model: 训练好的tomotopy模型。
    - top_n: 用于计算JSD的每个主题的top N个词。
    """
    try:
        num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
        if num_topics < 2:
            return 0.0

        # 1. 收集所有主题的top N词和概率，并建立一个共享词汇表
        topic_word_probs = []
        vocab = set()
        for k in range(num_topics):
            try:
                words = model.get_topic_words(k, top_n=top_n)
                if not words: continue
                topic_word_probs.append(dict(words))
                vocab.update([word for word, prob in words])
            except:
                continue
        
        if len(topic_word_probs) < 2:
            return 0.0

        vocab_list = sorted(list(vocab))
        vocab_map = {word: i for i, word in enumerate(vocab_list)}
        
        # 2. 将每个主题的词分布转换为对齐的概率向量
        aligned_probs = []
        for topic_dict in topic_word_probs:
            prob_vector = np.zeros(len(vocab_list))
            for word, prob in topic_dict.items():
                if word in vocab_map:
                    prob_vector[vocab_map[word]] = prob
            
            # 归一化，使其成为有效的概率分布
            if prob_vector.sum() > 1e-9:
                prob_vector /= prob_vector.sum()
            else:
                continue # 跳过无效的空主题
            aligned_probs.append(prob_vector)

        if len(aligned_probs) < 2:
            return 0.0

        # 3. 计算所有主题对之间的JSD
        jsd_values = []
        for i in range(len(aligned_probs)):
            for j in range(i + 1, len(aligned_probs)):
                p = aligned_probs[i]
                q = aligned_probs[j]
                jsd = jensenshannon(p, q, base=2)
                if not np.isnan(jsd):
                    jsd_values.append(jsd**2) # JSD距离通常使用JSD值的平方

        if not jsd_values:
            return 0.0
        
        return float(np.mean(jsd_values))

    except Exception as e:
        # print(f"Error calculating JSD: {e}")
        return 0.0

In [19]:
# ===== General Topic Analysis Function =====
def analyze_model_topics(model, model_name="Model", top_words=5, min_prob=0.01, max_display=10):
    """
    General topic analysis function - applicable to all topic models
    
    Functionality:
    - Extracts active topics
    - Displays topic words and weights
    - Calculates topic activity rate
    """
    print(f"\n🔍 {model_name} Topic Analysis (showing top {top_words} words):")
    print("=" * 80)
    
    active_topics = 0
    topic_info = []
    
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    
    for k in range(num_topics):
        try:
            topic_words = model.get_topic_words(k, top_n=top_words)
            if topic_words and topic_words[0][1] > min_prob:
                active_topics += 1
                words_str = ", ".join([f"{word}({prob:.3f})" for word, prob in topic_words[:5]])
                topic_info.append((k, topic_words[0][1], words_str))
                
                if active_topics <= max_display:
                    print(f"Topic {k:3d} (weight:{topic_words[0][1]:.3f}): {words_str}")
        except:
            continue
    
    if active_topics > max_display:
        print(f"... (and {active_topics - max_display} more active topics)")
    
    print(f"\n📊 {model_name} Topic Statistics:")
    print(f"   - Active topics: {active_topics}/{num_topics}")
    print(f"   - Topic activity rate: {active_topics/num_topics*100:.1f}%")
    
    return topic_info

In [20]:
def calculate_r_hat(chains_history):
    """
    根据多个MCMC链的后半部分历史记录计算R-hat（Gelman-Rubin诊断）值。
    
    参数:
    - chains_history (np.ndarray): 一个2D numpy数组，形状为 (M, N)，
      其中 M 是链的数量（运行次数），N 是每个链记录的迭代次数。
      数组中的值是对数似然（log-likelihood）。

    返回:
    - float: R-hat值。如果无法计算，则返回 np.nan。
    """
    # 仅使用后半部分的样本进行计算，这是标准做法
    num_iterations = chains_history.shape[1]
    start_index = num_iterations // 2
    
    if chains_history.shape[0] < 2 or (num_iterations - start_index) < 2:
        return np.nan

    chains_history = chains_history[:, start_index:]
    num_chains, num_used_iterations = chains_history.shape
    
    # 1. 计算每个链的均值
    chain_means = np.mean(chains_history, axis=1)
    
    # 2. 计算每个链的方差
    chain_variances = np.var(chains_history, axis=1, ddof=1)
    
    # 3. 计算链内方差的均值 (W)
    W = np.mean(chain_variances)
    
    # 4. 计算链间方差 (B)
    overall_mean = np.mean(chain_means)
    B = num_used_iterations / (num_chains - 1) * np.sum((chain_means - overall_mean)**2)
    
    # 5. 估计目标分布的方差 (Var_hat)
    var_hat = (1 - 1 / num_used_iterations) * W + (1 / num_used_iterations) * B
    
    if W == 0:
        return np.nan
        
    # 6. 计算R-hat
    r_hat = np.sqrt(var_hat / W)
    
    return r_hat

In [21]:
def calculate_weighted_coherence(model, corpus_docs, metrics=['c_v', 'c_npmi'], top_n=10, threshold=0.01):
    """
    计算按主题的文档覆盖率加权的 coherence 分数 (NPMI, C_v)。
    
    Args:
        model: 训练好的 tomotopy 模型。
        corpus_docs: 用于计算 coherence 的原始文档列表。
        metrics: 要计算的指标列表。
        top_n: 用于定义主题的 top N 个词。
        threshold: 判断一个主题在文档中是否“显著”的概率阈值。

    Returns:
        一个包含加权和未加权 coherence 分数的字典。
    """
    try:
        docs = list(corpus_docs)
        dictionary = Dictionary(docs)
        num_topics = getattr(model, 'k', 0) or getattr(model, 'num_topics', 0)
        
        if num_topics == 0:
            return {}

        # 1. 提取所有主题的 top words
        topics = []
        for k in range(num_topics):
            topic_words = [word for word, prob in model.get_topic_words(k, top_n=top_n)]
            if topic_words:
                topics.append(topic_words)
        
        if not topics:
            return {}

        # 2. 获取每个主题的权重 (文档覆盖数)
        topic_doc_counts = np.zeros(num_topics)
        for doc in model.docs:
            topic_dist = doc.get_topic_dist()
            for k, prob in enumerate(topic_dist):
                if prob > threshold:
                    topic_doc_counts[k] += 1
        
        total_docs_covered = np.sum(topic_doc_counts)
        
        results = {}
        for metric in metrics:
            cm = CoherenceModel(
                topics=topics,
                texts=docs,
                dictionary=dictionary,
                coherence=metric,
                processes=1
            )
            
            # 获取每个主题的分数
            per_topic_scores = cm.get_coherence_per_topic()
            
            # 计算简单平均值 (未加权)
            unweighted_avg = np.mean(per_topic_scores)
            results[f'unweighted_{metric}'] = unweighted_avg
            
            # 计算加权平均值
            if total_docs_covered > 0:
                weighted_avg = np.average(per_topic_scores, weights=topic_doc_counts)
                results[f'weighted_{metric}'] = weighted_avg
            else:
                results[f'weighted_{metric}'] = unweighted_avg # 如果没有权重，则退回到简单平均

        return results

    except Exception as e:
        print(f"计算加权Coherence时出错: {e}")
        return {}

# --- 如何使用 ---
# 假设 best_lda_model 是你训练好的模型
# weighted_scores = calculate_weighted_coherence(best_lda_model, abstract_list)
# print(weighted_scores)
# 输出可能像这样:
# {'unweighted_c_v': 0.45, 'weighted_c_v': 0.48, 'unweighted_c_npmi': 0.05, 'weighted_c_npmi': 0.06}

In [22]:
def tune_lda_model(
    docs, 
    k_range, 
    alpha_range, 
    eta_range, 
    num_runs=3, 
    seed=42, 
    max_iters=2000,
    convergence_patience=10,
    convergence_tolerance=1e-2,
    burn_in_iterations=1000  # <--- 新增 burn-in 参数
):
    """
    为 LDA 模型执行网格搜索以找到最佳超参数。
    此版本包含一个 burn-in 阶段，以提高收敛判断的稳定性。
    主要搜索主题数 k，但也支持 alpha 和 eta。
    使用加权Coherence作为主要评估指标之一。

    Args:
        docs: 文档语料库。
        k_range: 要测试的主题数 k 的列表。
        alpha_range: 要测试的 alpha 值的列表。
        eta_range: 要测试的 eta 值的列表。
        num_runs: 每个参数组合的运行次数。
        seed: 随机种子。
        max_iters: 最大迭代次数。
        convergence_patience: 判断收敛所需的连续检查次数。
        convergence_tolerance: 判断收敛的对数似然波动阈值。
        burn_in_iterations: 在开始检查收敛前的预热迭代次数。
    """
    all_run_results = []
    best_ll_per_word = -np.inf  # 基于LL选择最佳模型，越高越好
    best_params = {}
    best_model = None

    param_grid = list(itertools.product(k_range, alpha_range, eta_range))
    total_combinations = len(param_grid)
    
    print(f"🚀 开始为 LDA 进行网格搜索，共 {total_combinations} 种组合，每种运行 {num_runs} 次...")
    print(f"   (Burn-in: {burn_in_iterations} 次, 收敛检查: LL 范围 < {convergence_tolerance} ，持续 {convergence_patience} 次检查)")

    for i, (k, alpha, eta) in enumerate(param_grid, 1):
        print(f"\n--- 组合 {i}/{total_combinations}: 测试 K={k}, Alpha={alpha}, Eta={eta}，运行 {num_runs} 次 ---")
        
        for run_idx in range(num_runs):
            start_time = time.time()
            current_seed = seed + run_idx
            print(f"  - 运行 {run_idx + 1}/{num_runs} (seed={current_seed})...")
            
            try:
                # 初始化 tp.LDAModel
                model = tp.LDAModel(k=k, alpha=alpha, eta=eta, seed=current_seed)
                for doc in docs:
                    model.add_doc(doc)
                
                # 1. 执行 Burn-in 阶段 (不检查收敛)
                if burn_in_iterations > 0 and burn_in_iterations < max_iters:
                    model.train(burn_in_iterations, 1)

                # 2. 动态训练与收敛判断 (在 burn-in 之后开始)
                ll_history = []
                converged = False
                
                # 计算剩余的迭代次数
                remaining_iters = max_iters - burn_in_iterations
                final_iter = burn_in_iterations

                for step in range(0, remaining_iters, 10):
                    model.train(10, 1)
                    ll_history.append(model.ll_per_word)
                    final_iter += 10

                    if len(ll_history) > convergence_patience:
                        recent_ll = ll_history[-convergence_patience:]
                        if (np.max(recent_ll) - np.min(recent_ll)) < convergence_tolerance:
                            print(f"    => 在第 {final_iter} 次迭代时收敛")
                            converged = True
                            break
                
                # 如果循环正常结束但未收敛，将最终迭代次数设为最大值
                if not converged:
                    final_iter = max_iters

                # 评估指标
                ll_per_word = model.ll_per_word
                perplexity = math.exp(-ll_per_word)
                coherence_scores = calculate_weighted_coherence(model, docs, metrics=['c_v', 'c_npmi'])
                renyi_entropy = calculate_weighted_renyi_entropy(model)
                jsd_diversity = calculate_topic_diversity_jsd(model)
                
                run_result = {
                    'K': k, 'Alpha': alpha, 'Eta': eta,
                    'Run_Index': run_idx + 1,
                    'LL_per_word': ll_per_word,
                    'Weighted_NPMI': coherence_scores.get('weighted_c_npmi', 0),
                    'Weighted_Cv': coherence_scores.get('weighted_c_v', 0),
                    'Unweighted_NPMI': coherence_scores.get('unweighted_c_npmi', 0),
                    'Unweighted_Cv': coherence_scores.get('unweighted_c_v', 0),
                    'Perplexity': perplexity,
                    'Weighted_Renyi_Entropy': renyi_entropy,
                    'JSD_Diversity': jsd_diversity,
                    'Iterations': final_iter,
                    'Converged': "Yes" if converged else "No",
                    'Time_sec': time.time() - start_time
                }
                all_run_results.append(run_result)
                
                print(f"    => 结果: LL/word={run_result['LL_per_word']:.4f}, Weighted_NPMI={run_result['Weighted_NPMI']:.4f}, PPL={run_result['Perplexity']:.2f}, Renyi={run_result['Weighted_Renyi_Entropy']:.4f}, JSD={run_result['JSD_Diversity']:.4f}, Converged={run_result['Converged']}")

                # 基于LL/word来选择单次运行中的最佳模型
                if run_result['LL_per_word'] > best_ll_per_word:
                    best_ll_per_word = run_result['LL_per_word']
                    best_params = {'K': k, 'Alpha': alpha, 'Eta': eta}
                    best_model = model

            except Exception as e:
                print(f"    ❌ 运行 {run_idx + 1} 失败: {e}")

    results_df = pd.DataFrame(all_run_results)
    print("\n\n✅ LDA 调优完成!")
    print("--- 最终网格搜索结果 ---")
    print(results_df.to_string())
    
    if best_params:
        print(f"\n🏆 最佳单次运行参数 (基于LL/word): K={best_params['K']}, Alpha={best_params['Alpha']}, Eta={best_params['Eta']}，LL/word得分为 {best_ll_per_word:.4f}")
    else:
        print("\n❌ 未完成任何成功的运行。")

    return results_df, best_model, best_params

In [23]:
# ===== LDA 网格搜索实例 =====
print("\n🎯 开始为 LDA 模型进行网格搜索...")
print("=" * 60)

# 1. 定义超参数范围
# #    - K (主题数): 定义一个你认为可能合理的范围
k_range_to_test = [40, 60, 80, 100, 120, 140, 160, 180, 200]

#    - Alpha (文档-主题分布的先验): 较小的值意味着每个文档由少数几个主题主导
alpha_range_to_test = [0.1]

#    - Eta (主题-词分布的先验): 较小的值意味着每个主题由少数几个词主导
eta_range_to_test = [0.01]

# 2. 确保文档已准备好
lda_docs = abstract_list

# 3. 运行网格搜索函数
#    注意：这里我们将 num_runs 设置为 5，以便后续进行裁剪平均分析
lda_results_df, best_lda_model, best_lda_params = tune_lda_model(
    docs=lda_docs,
    k_range=k_range_to_test,
    alpha_range=alpha_range_to_test,
    eta_range=eta_range_to_test,
    num_runs=5,  # 为每个参数组合运行5次
    max_iters=3000
)

# 4. 保存详细的运行结果，以便将来分析
#    文件名可以自定义，以区分不同的实验
results_filename = './data/model_result/lda_tuning_results_weighted.csv'
print(f"\n💾 将详细结果保存到: {results_filename}")
lda_results_df.to_csv(results_filename, index=False, encoding='utf-8-sig')

print("\n🎉 LDA 网格搜索实例运行完成！")


🎯 开始为 LDA 模型进行网格搜索...
🚀 开始为 LDA 进行网格搜索，共 9 种组合，每种运行 5 次...
   (Burn-in: 1000 次, 收敛检查: LL 范围 < 0.01 ，持续 10 次检查)

--- 组合 1/9: 测试 K=40, Alpha=0.1, Eta=0.01，运行 5 次 ---
  - 运行 1/5 (seed=42)...
    => 在第 1540 次迭代时收敛
    => 结果: LL/word=-6.9805, Weighted_NPMI=0.0423, PPL=1075.43, Renyi=3.6064, JSD=0.9537, Converged=Yes
  - 运行 2/5 (seed=43)...
    => 在第 1690 次迭代时收敛
    => 结果: LL/word=-6.9649, Weighted_NPMI=0.0360, PPL=1058.77, Renyi=3.5769, JSD=0.9535, Converged=Yes
  - 运行 3/5 (seed=44)...
    => 在第 1110 次迭代时收敛
    => 结果: LL/word=-6.9553, Weighted_NPMI=0.0404, PPL=1048.72, Renyi=3.5554, JSD=0.9570, Converged=Yes
  - 运行 4/5 (seed=45)...
    => 在第 1450 次迭代时收敛
    => 结果: LL/word=-6.9546, Weighted_NPMI=0.0522, PPL=1047.98, Renyi=3.5732, JSD=0.9569, Converged=Yes
  - 运行 5/5 (seed=46)...
    => 在第 1210 次迭代时收敛
    => 结果: LL/word=-6.9893, Weighted_NPMI=0.0407, PPL=1085.00, Renyi=3.6337, JSD=0.9543, Converged=Yes

--- 组合 2/9: 测试 K=60, Alpha=0.1, Eta=0.01，运行 5 次 ---
  - 运行 1/5 (seed=42)...
    => 在第 122

In [24]:
# ===== 分析调优结果：仅对收敛的运行求均值 =====
print("\n\n📊 开始分析LDA调优结果 (仅限收敛的运行)...")
print("=" * 80)

try:
    # 尝试使用内存中的 lda_results_df，如果不存在则从文件加载
    if 'lda_results_df' not in locals():
        results_filename = './data/model_result/lda_tuning_results_weighted.csv'
        print(f"从 '{results_filename}' 加载结果...")
        lda_results_df = pd.read_csv(results_filename)

    # 1. 筛选出所有成功收敛的运行
    converged_runs_df = lda_results_df[lda_results_df['Converged'] == 'Yes'].copy()
    
    if converged_runs_df.empty:
        print("\n❌ 警告: 在所有运行中，没有一次是成功收敛的。无法进行分析。")
    else:
        print(f"原始运行总数: {len(lda_results_df)}, 成功收敛的运行数: {len(converged_runs_df)}")
        
        # 2. 按参数组合进行分组，并对每个指标直接求平均值
        aggregated_results = converged_runs_df.groupby(['K', 'Alpha', 'Eta']).agg(
        Avg_Weighted_NPMI=('Weighted_NPMI', 'mean'),
        Avg_Weighted_Cv=('Weighted_Cv', 'mean'),
        Avg_Perplexity=('Perplexity', 'mean'),
        Avg_Weighted_Renyi=('Weighted_Renyi_Entropy', 'mean'),
        Avg_JSD_Diversity=('JSD_Diversity', 'mean'),
        Converged_Runs=('Run_Index', 'count')  # 统计每个组合收敛的次数
        ).reset_index()

        # --- 排名 1: 按平均加权NPMI降序排序 (越高越好) ---
        results_by_npmi = aggregated_results.sort_values(by='Avg_Weighted_NPMI', ascending=False)

        print("\n✅ LDA 调优结果 (排名依据: 平均加权NPMI - 越高越好):")
        print(results_by_npmi.to_string())

        # 提取并打印基于NPMI的最佳参数组合
        if not results_by_npmi.empty:
            best_params_npmi = results_by_npmi.iloc[0]
            print("\n\n🏆 最佳参数组合 (基于平均加权NPMI):")
            print(f"   K: {int(best_params_npmi['K'])}")
            print(f"   Alpha: {best_params_npmi['Alpha']}")
            print(f"   Eta: {best_params_npmi['Eta']}")
            print(f"   => 平均加权 NPMI: {best_params_npmi['Avg_Weighted_NPMI']:.4f}")
            print(f"   => 平均加权 Renyi熵: {best_params_npmi['Avg_Weighted_Renyi']:.4f}")
            print(f"   => 平均 JSD多样性: {best_params_npmi['Avg_JSD_Diversity']:.4f}")
            print(f"   (基于 {int(best_params_npmi['Converged_Runs'])} 次收敛运行的平均结果)")

        # --- 排名 2: 按平均困惑度升序排序 (越低越好) ---
        results_by_perplexity = aggregated_results.sort_values(by='Avg_Perplexity', ascending=True)

        print("\n\n✅ LDA 调优结果 (排名依据: 平均困惑度 - 越低越好):")
        print(results_by_perplexity.to_string())

        # 提取并打印基于Perplexity的最佳参数组合
        if not results_by_perplexity.empty:
            best_params_ppl = results_by_perplexity.iloc[0]
            print("\n\n🏆 最佳参数组合 (基于平均困惑度):")
            print(f"   K: {int(best_params_ppl['K'])}")
            print(f"   Alpha: {best_params_ppl['Alpha']}")
            print(f"   Eta: {best_params_ppl['Eta']}")
            print(f"   => 平均困惑度: {best_params_ppl['Avg_Perplexity']:.4f}")
            print(f"   => 平均加权 Renyi熵: {best_params_ppl['Avg_Weighted_Renyi']:.4f}")
            print(f"   => 平均 JSD多样性: {best_params_ppl['Avg_JSD_Diversity']:.4f}")
            print(f"   (基于 {int(best_params_ppl['Converged_Runs'])} 次收敛运行的平均结果)")

except FileNotFoundError:
    # 更新了文件名以匹配保存时的文件名
    results_filename = './data/model_result/lda_tuning_results_weighted.csv'
    print(f"\n❌ 错误: 未找到 '{results_filename}' 文件。")
    print("请确保第一阶段的网格搜索已成功运行并保存了结果。")
except Exception as e:
    print(f"\n❌ 处理结果时发生错误: {e}")



📊 开始分析LDA调优结果 (仅限收敛的运行)...
原始运行总数: 45, 成功收敛的运行数: 42

✅ LDA 调优结果 (排名依据: 平均加权NPMI - 越高越好):
     K  Alpha   Eta  Avg_Weighted_NPMI  Avg_Weighted_Cv  Avg_Perplexity  Avg_Weighted_Renyi  Avg_JSD_Diversity  Converged_Runs
0   40    0.1  0.01           0.042313         0.500168     1063.181026            3.589104           0.955063               5
1   60    0.1  0.01           0.041661         0.506731     1098.963622            3.371992           0.966408               5
2   80    0.1  0.01           0.038096         0.503990     1099.961942            3.183809           0.972645               5
3  100    0.1  0.01           0.031343         0.496881     1140.528277            3.085951           0.974478               5
4  120    0.1  0.01           0.028914         0.501070     1139.920207            3.003056           0.976692               5
5  140    0.1  0.01           0.022770         0.494065     1184.472278            2.928057           0.978142               4
6  160    0.1  0.01 

In [25]:
# ===== LDA 网格搜索实例 =====
print("\n🎯 开始为 LDA 模型进行网格搜索...")
print("=" * 60)

# 1. 定义超参数范围
# #    - K (主题数): 定义一个你认为可能合理的范围
k_range_to_test = [10,20]

#    - Alpha (文档-主题分布的先验): 较小的值意味着每个文档由少数几个主题主导
alpha_range_to_test = [0.1]

#    - Eta (主题-词分布的先验): 较小的值意味着每个主题由少数几个词主导
eta_range_to_test = [0.01]

# 2. 确保文档已准备好
lda_docs = abstract_list

# 3. 运行网格搜索函数
#    注意：这里我们将 num_runs 设置为 5，以便后续进行裁剪平均分析
lda_results_df, best_lda_model, best_lda_params = tune_lda_model(
    docs=lda_docs,
    k_range=k_range_to_test,
    alpha_range=alpha_range_to_test,
    eta_range=eta_range_to_test,
    num_runs=5,  # 为每个参数组合运行5次
    max_iters=3000
)

# 4. 保存详细的运行结果，以便将来分析
#    文件名可以自定义，以区分不同的实验
results_filename = './data/model_result/lda_tuning_results_weighted_added.csv'
print(f"\n💾 将详细结果保存到: {results_filename}")
lda_results_df.to_csv(results_filename, index=False, encoding='utf-8-sig')

print("\n🎉 LDA 网格搜索实例运行完成！")


🎯 开始为 LDA 模型进行网格搜索...
🚀 开始为 LDA 进行网格搜索，共 2 种组合，每种运行 5 次...
   (Burn-in: 1000 次, 收敛检查: LL 范围 < 0.01 ，持续 10 次检查)

--- 组合 1/2: 测试 K=10, Alpha=0.1, Eta=0.01，运行 5 次 ---
  - 运行 1/5 (seed=42)...
    => 在第 1260 次迭代时收敛
    => 结果: LL/word=-6.9161, Weighted_NPMI=0.0263, PPL=1008.40, Renyi=4.4271, JSD=0.9012, Converged=Yes
  - 运行 2/5 (seed=43)...
    => 在第 1300 次迭代时收敛
    => 结果: LL/word=-6.9380, Weighted_NPMI=0.0283, PPL=1030.66, Renyi=4.4699, JSD=0.8935, Converged=Yes
  - 运行 3/5 (seed=44)...
    => 在第 1500 次迭代时收敛
    => 结果: LL/word=-6.8970, Weighted_NPMI=0.0342, PPL=989.30, Renyi=4.4368, JSD=0.8920, Converged=Yes
  - 运行 4/5 (seed=45)...
    => 在第 1190 次迭代时收敛
    => 结果: LL/word=-6.9258, Weighted_NPMI=0.0195, PPL=1018.22, Renyi=4.4593, JSD=0.9010, Converged=Yes
  - 运行 5/5 (seed=46)...
    => 在第 1120 次迭代时收敛
    => 结果: LL/word=-6.9093, Weighted_NPMI=0.0383, PPL=1001.59, Renyi=4.4825, JSD=0.8804, Converged=Yes

--- 组合 2/2: 测试 K=20, Alpha=0.1, Eta=0.01，运行 5 次 ---
  - 运行 1/5 (seed=42)...
    => 在第 1120

In [26]:
# ===== 分析调优结果：仅对收敛的运行求均值 =====
print("\n\n📊 开始分析LDA调优结果 (仅限收敛的运行)...")
print("=" * 80)

try:
    # 尝试使用内存中的 lda_results_df，如果不存在则从文件加载
    if 'lda_results_df' not in locals():
        results_filename = './data/model_result/lda_tuning_results_weighted_added.csv'
        print(f"从 '{results_filename}' 加载结果...")
        lda_results_df = pd.read_csv(results_filename)

    # 1. 筛选出所有成功收敛的运行
    converged_runs_df = lda_results_df[lda_results_df['Converged'] == 'Yes'].copy()
    
    if converged_runs_df.empty:
        print("\n❌ 警告: 在所有运行中，没有一次是成功收敛的。无法进行分析。")
    else:
        print(f"原始运行总数: {len(lda_results_df)}, 成功收敛的运行数: {len(converged_runs_df)}")
        
        # 2. 按参数组合进行分组，并对每个指标直接求平均值
        aggregated_results = converged_runs_df.groupby(['K', 'Alpha', 'Eta']).agg(
        Avg_Weighted_NPMI=('Weighted_NPMI', 'mean'),
        Avg_Weighted_Cv=('Weighted_Cv', 'mean'),
        Avg_Perplexity=('Perplexity', 'mean'),
        Avg_Weighted_Renyi=('Weighted_Renyi_Entropy', 'mean'),
        Avg_JSD_Diversity=('JSD_Diversity', 'mean'),
        Converged_Runs=('Run_Index', 'count')  # 统计每个组合收敛的次数
        ).reset_index()

        # --- 排名 1: 按平均加权NPMI降序排序 (越高越好) ---
        results_by_npmi = aggregated_results.sort_values(by='Avg_Weighted_NPMI', ascending=False)

        print("\n✅ LDA 调优结果 (排名依据: 平均加权NPMI - 越高越好):")
        print(results_by_npmi.to_string())

        # 提取并打印基于NPMI的最佳参数组合
        if not results_by_npmi.empty:
            best_params_npmi = results_by_npmi.iloc[0]
            print("\n\n🏆 最佳参数组合 (基于平均加权NPMI):")
            print(f"   K: {int(best_params_npmi['K'])}")
            print(f"   Alpha: {best_params_npmi['Alpha']}")
            print(f"   Eta: {best_params_npmi['Eta']}")
            print(f"   => 平均加权 NPMI: {best_params_npmi['Avg_Weighted_NPMI']:.4f}")
            print(f"   => 平均加权 Renyi熵: {best_params_npmi['Avg_Weighted_Renyi']:.4f}")
            print(f"   => 平均 JSD多样性: {best_params_npmi['Avg_JSD_Diversity']:.4f}")
            print(f"   (基于 {int(best_params_npmi['Converged_Runs'])} 次收敛运行的平均结果)")

        # --- 排名 2: 按平均困惑度升序排序 (越低越好) ---
        results_by_perplexity = aggregated_results.sort_values(by='Avg_Perplexity', ascending=True)

        print("\n\n✅ LDA 调优结果 (排名依据: 平均困惑度 - 越低越好):")
        print(results_by_perplexity.to_string())

        # 提取并打印基于Perplexity的最佳参数组合
        if not results_by_perplexity.empty:
            best_params_ppl = results_by_perplexity.iloc[0]
            print("\n\n🏆 最佳参数组合 (基于平均困惑度):")
            print(f"   K: {int(best_params_ppl['K'])}")
            print(f"   Alpha: {best_params_ppl['Alpha']}")
            print(f"   Eta: {best_params_ppl['Eta']}")
            print(f"   => 平均困惑度: {best_params_ppl['Avg_Perplexity']:.4f}")
            print(f"   => 平均加权 Renyi熵: {best_params_ppl['Avg_Weighted_Renyi']:.4f}")
            print(f"   => 平均 JSD多样性: {best_params_ppl['Avg_JSD_Diversity']:.4f}")
            print(f"   (基于 {int(best_params_ppl['Converged_Runs'])} 次收敛运行的平均结果)")

except FileNotFoundError:
    # 更新了文件名以匹配保存时的文件名
    results_filename = './data/model_result/lda_tuning_results_weighted.csv'
    print(f"\n❌ 错误: 未找到 '{results_filename}' 文件。")
    print("请确保第一阶段的网格搜索已成功运行并保存了结果。")
except Exception as e:
    print(f"\n❌ 处理结果时发生错误: {e}")



📊 开始分析LDA调优结果 (仅限收敛的运行)...
原始运行总数: 10, 成功收敛的运行数: 10

✅ LDA 调优结果 (排名依据: 平均加权NPMI - 越高越好):
    K  Alpha   Eta  Avg_Weighted_NPMI  Avg_Weighted_Cv  Avg_Perplexity  Avg_Weighted_Renyi  Avg_JSD_Diversity  Converged_Runs
1  20    0.1  0.01           0.035733         0.507421     1029.435962            4.033555           0.929779               5
0  10    0.1  0.01           0.029324         0.500811     1009.634633            4.455119           0.893625               5


🏆 最佳参数组合 (基于平均加权NPMI):
   K: 20
   Alpha: 0.1
   Eta: 0.01
   => 平均加权 NPMI: 0.0357
   => 平均加权 Renyi熵: 4.0336
   => 平均 JSD多样性: 0.9298
   (基于 5 次收敛运行的平均结果)


✅ LDA 调优结果 (排名依据: 平均困惑度 - 越低越好):
    K  Alpha   Eta  Avg_Weighted_NPMI  Avg_Weighted_Cv  Avg_Perplexity  Avg_Weighted_Renyi  Avg_JSD_Diversity  Converged_Runs
0  10    0.1  0.01           0.029324         0.500811     1009.634633            4.455119           0.893625               5
1  20    0.1  0.01           0.035733         0.507421     1029.435962            4.0335

In [27]:
# ===== LDA 网格搜索实例 =====
print("\n🎯 开始为 LDA 模型进行网格搜索...")
print("=" * 60)

# 1. 定义新的超参数范围
#    - 在上一轮找到的最佳K=80附近进行精细化搜索
k_range_to_test_stage2 = [40,60,80]

#    - 探索不同的alpha值，包括自动优化选项
alpha_range_to_test_stage2 = [0.05,0.1,0.5,1,5,10,20]

#    - 暂时保持eta不变
eta_range_to_test_stage2 = [0.01,0.1,0.5,1,2]

# 2. 确保文档已准备好
lda_docs = abstract_list

# 3. 运行网格搜索函数
#    注意：这里我们将 num_runs 设置为 5，以便后续进行裁剪平均分析
lda_results_df, best_lda_model, best_lda_params = tune_lda_model(
    docs=lda_docs,
    k_range=k_range_to_test_stage2,
    alpha_range=alpha_range_to_test_stage2,
    eta_range=eta_range_to_test_stage2,
    num_runs=5,  # 为每个参数组合运行5次
    max_iters=3000
)

# 4. 保存详细的运行结果，以便将来分析
#    文件名可以自定义，以区分不同的实验
results_filename = './data/model_result/lda_tuning_results_weighted_stage2.csv'
print(f"\n💾 将详细结果保存到: {results_filename}")
lda_results_df.to_csv(results_filename, index=False, encoding='utf-8-sig')

print("\n🎉 LDA 网格搜索实例运行完成！")


🎯 开始为 LDA 模型进行网格搜索...
🚀 开始为 LDA 进行网格搜索，共 105 种组合，每种运行 5 次...
   (Burn-in: 1000 次, 收敛检查: LL 范围 < 0.01 ，持续 10 次检查)

--- 组合 1/105: 测试 K=40, Alpha=0.05, Eta=0.01，运行 5 次 ---
  - 运行 1/5 (seed=42)...
    => 在第 1230 次迭代时收敛
    => 结果: LL/word=-6.9909, Weighted_NPMI=0.0375, PPL=1086.66, Renyi=3.5983, JSD=0.9551, Converged=Yes
  - 运行 2/5 (seed=43)...
    => 在第 1420 次迭代时收敛
    => 结果: LL/word=-6.9525, Weighted_NPMI=0.0438, PPL=1045.78, Renyi=3.6206, JSD=0.9528, Converged=Yes
  - 运行 3/5 (seed=44)...
    => 在第 1260 次迭代时收敛
    => 结果: LL/word=-6.9474, Weighted_NPMI=0.0442, PPL=1040.47, Renyi=3.5887, JSD=0.9550, Converged=Yes
  - 运行 4/5 (seed=45)...
    => 在第 1850 次迭代时收敛
    => 结果: LL/word=-6.9650, Weighted_NPMI=0.0428, PPL=1058.94, Renyi=3.6558, JSD=0.9507, Converged=Yes
  - 运行 5/5 (seed=46)...
    => 在第 1310 次迭代时收敛
    => 结果: LL/word=-6.9731, Weighted_NPMI=0.0414, PPL=1067.55, Renyi=3.6720, JSD=0.9546, Converged=Yes

--- 组合 2/105: 测试 K=40, Alpha=0.05, Eta=0.1，运行 5 次 ---
  - 运行 1/5 (seed=42)...
    =>